In [1]:
# Abstraction Steering (Search ver.)
# ----------------------------------
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from sorl.steer import StackedAbstractionWrapperV8

MODEL_NAME = "Qwen/Qwen3-0.6B"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Load model + tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

D_MODEL = model.config.hidden_size
N_LAYERS = model.config.num_hidden_layers
print(f"Model: {MODEL_NAME} | D={D_MODEL} | layers={N_LAYERS}")

# Wrap with V8 (STE-trainable routing)
C_SIZE = 4
L = 4
INJECT_LAYERS = [N_LAYERS // 2]  # mid layer
SCALE = 0.5

# wrapper = StackedAbstractionWrapperV8(
#     model, C_SIZE=C_SIZE, D_MODEL=D_MODEL,
#     inject_layers=INJECT_LAYERS, scale=SCALE, L=L,
#     code_position="first",
# )
# model = model.to(DEVICE)
# wrapper.routing_proj.to(DEVICE)
# wrapper.steering_emb.to(DEVICE)

# n_steer = sum(p.numel() for p in wrapper.routing_proj.parameters()) + \
#           sum(p.numel() for p in wrapper.steering_emb.parameters())
# print(f"V8 wrapper: C={C_SIZE}, L={L}, scale={SCALE}, layers={INJECT_LAYERS}")
# print(f"Steering params: {n_steer/1e3:.1f}K")

/Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model: Qwen/Qwen3-0.6B | D=1024 | layers=28


In [2]:
# ── V9 Wrapper + SoRL-style Training Loop ──────────────────────────────
# Key idea: steering codes are determined by a learned abs_proj head.
# "Search" = try N random routing perturbations, keep the one with lowest CE.
# Then train abs_proj + steering_emb on the winning configuration.

import time, os, json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sorl.steer import StackedAbstractionWrapperV9
from data.pt_dataset import get_dataset

# ── Config ──
DATASET = "scienceqa"
MAX_LENGTH = 512
BATCH_SIZE = 2
GRAD_ACCUM = 4
NUM_EPOCHS = 1
LR = 1e-5
STEER_LR = 5e-2
WARMUP_STEPS = 50
MAX_GRAD_NORM = 1.0
LOG_EVERY = 10
C_SIZE = 4
L_CHUNK = 4
SCALE = 0.5
INJECT_LAYERS = [14]
NUM_ROLLOUTS = 4  # search: try N routing perturbations per batch

# ── Build V9 wrapper (reuse model/tokenizer from cell 0) ──
wrapper_v9 = StackedAbstractionWrapperV9(
    model, C_SIZE=C_SIZE, D_MODEL=D_MODEL,
    inject_layers=INJECT_LAYERS, scale=SCALE, L=L_CHUNK,
    code_position="first",
)
wrapper_v9.abs_proj.to(DEVICE)
wrapper_v9.steering_emb.to(DEVICE)

steer_params = wrapper_v9.get_steer_params()
n_steer = sum(p.numel() for p in steer_params)
n_model = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"V9: C={C_SIZE}, L={L_CHUNK}, scale={SCALE}, layers={INJECT_LAYERS}")
print(f"Trainable: model={n_model/1e6:.1f}M  steer={n_steer/1e3:.1f}K (abs_proj + steering_emb)")

# ── Dataset ──
train_ds = get_dataset(DATASET, split="train", tokenizer=tokenizer, max_length=MAX_LENGTH)
print(f"Train: {len(train_ds)} samples")

V9: C=4, L=4, scale=0.5, layers=[14]
Trainable: model=751.6M  steer=8.2K (abs_proj + steering_emb)
Train: 6508 samples


In [3]:
# Hyp 1. Initialization for steering emb matters, for STE
# Hyp 2. Including "tempted sampling" during training to improve performance
# Hyp 3. Including "search" to improve abstraction routing beyond being static, adopting v1 methods
#        - abs_route(h.detach()) -> do not joint training policy & rep
#        - abs_route(h) -> joint training policy & rep

In [ ]:
# ── SoRL-style Training Loop for V9 ─────────────────────────────────────
# Phase per batch:
#   1. Base CE loss (no steering, scale=0) — the "SFT baseline" for this batch
#   2. Search: repeat batch N times, forward(temperature=T, reduction='none'),
#      each copy samples different codes → select best per original sample
#   3. Train: forward(forced_codes=best_codes) — exact winning routing
#      - ce_loss:   standard next-token CE (with winning steering)
#      - info_gain: CE(steered) - CE(base) — negative means steering helped
#      - abs_loss:  CE(abs_proj logits, best_codes) — teach routing to reproduce search result
#      - steer_reg: L2 on steering embeddings

from collections import defaultdict
from sorl.sorl_trainer import VariableZipfian2gramLoss as ZipLoss

ALPHA_INFO = 1.0
ALPHA_ABS  = 0.5
ALPHA_REG  = 0.01
SEARCH_TEMP = 1.0

zipf_loss_fn = ZipLoss(vocab_size=tokenizer.vocab_size)

def collate_fn(batch):
    out = {}
    for k in batch[0]:
        vals = [b[k] for b in batch]
        if isinstance(vals[0], torch.Tensor):
            out[k] = torch.stack(vals)
        else:
            out[k] = torch.tensor(vals)
    return out

dataloader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_fn, num_workers=0, pin_memory=False,
)
total_steps = len(dataloader) * NUM_EPOCHS // GRAD_ACCUM

param_groups = [
    {'params': [p for p in model.parameters() if p.requires_grad], 'lr': LR},
    {'params': steer_params, 'lr': STEER_LR},
]
optimizer = torch.optim.AdamW(param_groups, weight_decay=0.01)

def get_lr(step, total, warmup, base_lr):
    if step < warmup:
        return base_lr * step / max(warmup, 1)
    frac = (step - warmup) / max(total - warmup, 1)
    return base_lr * 0.5 * (1 + torch.cos(torch.tensor(frac * 3.14159)).item())

# ── Training ──
history = defaultdict(list)
model.train()
global_step = 0
t_start = time.time()

for epoch in range(NUM_EPOCHS):
    for batch_idx, batch in enumerate(dataloader):
        input_ids = batch['input_ids'].to(DEVICE)
        attn = batch['attention_mask'].to(DEVICE)
        prompt_len = batch['prompt_len'].to(DEVICE)
        B = input_ids.size(0)

        # Labels: mask padding + prompt
        labels = input_ids.clone()
        labels[attn == 0] = -100
        si = torch.arange(labels.size(1), device=DEVICE).unsqueeze(0)
        labels[si < prompt_len.unsqueeze(1)] = -100

        # ── Phase 1: Base CE (no steering) ──
        old_scale = wrapper_v9.scale
        wrapper_v9.scale = 0.0
        with torch.no_grad():
            base_out = wrapper_v9(input_ids, attn, labels)
            base_ce = base_out.loss.item()
        wrapper_v9.scale = old_scale

        # ── Phase 2: Search — parallel rollouts via temperature sampling ──
        N = NUM_ROLLOUTS
        rep_ids = input_ids.repeat_interleave(N, dim=0)    # (B*N, S)
        rep_attn = attn.repeat_interleave(N, dim=0)
        rep_labels = labels.repeat_interleave(N, dim=0)

        with torch.no_grad():
            rep_out = wrapper_v9(rep_ids, rep_attn, rep_labels,
                                temperature=SEARCH_TEMP, reduction='none')

            # Per-sample loss → (B, N) → pick best rollout per sample
            per_sample_loss = rep_out.per_sample_loss.view(B, N)
            best_idx = per_sample_loss.argmin(dim=-1)  # (B,)

            # Extract winning codes
            all_codes = wrapper_v9._last_chunk_codes.view(B, N, -1)
            best_codes = all_codes[torch.arange(B, device=DEVICE), best_idx]

            del rep_out

        # ── Phase 3: Train with forced_codes = search winners ──
        out = wrapper_v9(input_ids, attn, labels, forced_codes=best_codes)
        ce_loss = out.loss

        info_gain = ce_loss - base_ce # information gain
        abs_loss_val = wrapper_v9.compute_abs_loss(target_codes=best_codes) # memorize good abstraction choice
        zipf_loss = zipf_loss_fn(wrapper_v9._last_routing_logits) # route to diverse abstractions

        loss = ce_loss + ALPHA_INFO * info_gain + ALPHA_ABS * abs_loss_val + ALPHA_REG * zipf_loss
        loss = loss / GRAD_ACCUM
        loss.backward()

        if (batch_idx + 1) % GRAD_ACCUM == 0:
            if MAX_GRAD_NORM > 0:
                all_params = list(model.parameters()) + steer_params
                torch.nn.utils.clip_grad_norm_(all_params, MAX_GRAD_NORM)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

            lr = get_lr(global_step, total_steps, WARMUP_STEPS, LR)
            optimizer.param_groups[0]['lr'] = lr
            optimizer.param_groups[1]['lr'] = lr * (STEER_LR / max(LR, 1e-10))

        # Logging
        total_loss = loss.item() * GRAD_ACCUM
        if (batch_idx + 1) % LOG_EVERY == 0:
            steer_norm = sum(p.float().norm(dim=-1).mean().item()
                             for p in wrapper_v9.steering_emb.parameters())
            abs_l = abs_loss_val.item()
            ig = info_gain.item() if torch.is_tensor(info_gain) else info_gain
            print(f"ep {epoch + (batch_idx+1)/len(dataloader):.3f} | "
                  f"step {global_step}/{total_steps} | "
                  f"loss={total_loss:.4f} base={base_ce:.4f} "
                  f"info={ig:.4f} abs={abs_l:.4f} "
                  f"steer={steer_norm:.4f} | "
                  f"lr={optimizer.param_groups[0]['lr']:.2e}")
            history['step'].append(global_step)
            history['loss'].append(total_loss)
            history['base_ce'].append(base_ce)
            history['info_gain'].append(ig)
            history['abs_loss'].append(abs_l)

        del loss, out
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print(f"\nTraining done! {global_step} steps")

In [ ]:
# ── Visualize training curves ──
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

axes[0].plot(history['step'], history['loss'], 'b-', alpha=0.7)
axes[0].set_title('Total Loss')
axes[0].set_xlabel('Step')

axes[1].plot(history['step'], history['base_ce'], 'r-', alpha=0.7)
axes[1].set_title('Base CE (no steering)')
axes[1].set_xlabel('Step')

axes[2].plot(history['step'], history['info_gain'], 'g-', alpha=0.7)
axes[2].axhline(y=0, color='k', linestyle='--', alpha=0.3)
axes[2].set_title('Info Gain (neg=helps)')
axes[2].set_xlabel('Step')

axes[3].plot(history['step'], history['abs_loss'], 'm-', alpha=0.7)
axes[3].set_title('Abs Loss (routing prediction)')
axes[3].set_xlabel('Step')

plt.tight_layout()
plt.show()

# Show steering embedding norms + routing head
with torch.no_grad():
    w = wrapper_v9.steering_emb.weight.float()
    print(f"Steering emb norms: {[f'{n:.4f}' for n in w.norm(dim=-1).tolist()]}")
    print(f"abs_proj weight norm: {wrapper_v9.abs_proj.weight.float().norm():.4f}")